In [3]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/final_panel_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/final_panel_data.csv'

In [ ]:
import matplotlib.pyplot as plt\nimport seaborn as sns\n\nsparse_cols = df.columns[df.isna().any()].tolist()\nprint(f\"Columns with missing values ({len(sparse_cols)}): {sparse_cols}\")\n\nmissing_pct_by_country = (\n    df.groupby('isocode')[sparse_cols]\n    .apply(lambda group: group.isna().mean() * 100)\n    .sort_index()\n)\n\ncountry_missing_summary = pd.DataFrame({\n    'countries_with_any_missing': (missing_pct_by_country.gt(0)).sum(),\n    'countries_with_complete_coverage': (missing_pct_by_country.eq(0)).sum(),\n}).sort_values('countries_with_any_missing', ascending=False)\n\nprint('\\nCountry coverage summary by sparse column:')\nprint(country_missing_summary)\n\npartial_missingness = (\n    missing_pct_by_country.stack()\n    .rename('percent_missing')\n    .reset_index()\n    .query('0 < percent_missing < 100')\n    .sort_values(['percent_missing', 'isocode', 'level_1'], ascending=[False, True, True])\n    .rename(columns={'level_1': 'column'})\n)\n\nprint('\\nCountry-column pairs with partial missingness (1%-99%):')\nif partial_missingness.empty:\n    print('None')\nelse:\n    print(partial_missingness.to_string(index=False))\n\nn_countries = len(missing_pct_by_country)\nfigsize = (max(len(sparse_cols) * 1.2, 6), max(n_countries * 0.35, 3))\n\nplt.figure(figsize=figsize)\nsns.heatmap(\n    missing_pct_by_country,\n    cmap=sns.light_palette('darkred', as_cmap=True),\n    vmin=0,\n    vmax=100,\n    annot=n_countries < 20,\n    fmt='.1f',\n    cbar_kws={'label': 'Percent Missing'},\n)\nplt.title('Missing Data Coverage by Country (%)')\nplt.xlabel('Column')\nplt.ylabel('ISO Country Code')\nplt.tight_layout()\nplt.show()